In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [2]:
ARQUIVO = "CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"

df = pd.read_excel(ARQUIVO, sheet_name="Municípios", header=2)

# Remove colunas vazias criadas por células mescladas no Excel
df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed")])

# Remove linhas de nota de rodapé / fonte no final da planilha (não têm UF nem município)
df = df.dropna(subset=["UF", "NOME DO MUNICÍPIO"])

# Renomeia colunas para nomes mais simples de trabalhar
df = df.rename(columns={
    "População Município 2010\n(Sinopse)": "Populacao_2010_Sinopse",
    "População 2010 (Alterações de Limites até 2022)1": "Populacao_2010_Compatibilizada",
    "População Censo 2022": "Populacao_2022",
    "NOME DO MUNICÍPIO": "Municipio",
    "COD. UF": "Cod_UF",
    "COD. MUNIC": "Cod_Munic",
})

print(df.shape)
df.head()

(5570, 7)


,UF,Cod_UF,Cod_Munic,Municipio,Populacao_2010_Sinopse,Populacao_2010_Compatibilizada,Populacao_2022
0,RO,11.0,15.0,Alta Floresta D'Oeste,24392.0,24392.0,21494.0
1,RO,11.0,23.0,Ariquemes,90353.0,90353.0,96833.0
2,RO,11.0,31.0,Cabixi,6313.0,6313.0,5351.0
3,RO,11.0,49.0,Cacoal,78574.0,78574.0,86887.0
4,RO,11.0,56.0,Cerejeiras,17029.0,17029.0,15890.0


In [3]:
pop_estado = (
    df.groupby("UF")[["Populacao_2010_Compatibilizada", "Populacao_2022"]]
    .sum()
    .reset_index()
)

pop_estado.head()

,UF,Populacao_2010_Compatibilizada,Populacao_2022
0,AC,733559.0,830018.0
1,AL,3120887.0,3127683.0
2,AM,3483985.0,3941613.0
3,AP,669526.0,733759.0
4,BA,14017071.0,14141626.0


In [4]:
pop_estado["Crescimento_Absoluto"] = pop_estado["Populacao_2022"] - pop_estado["Populacao_2010_Compatibilizada"]
pop_estado["Crescimento_Percentual"] = (
    pop_estado["Crescimento_Absoluto"] / pop_estado["Populacao_2010_Compatibilizada"] * 100
).round(2)

pop_estado = pop_estado.sort_values("Crescimento_Absoluto", ascending=False).reset_index(drop=True)

pop_estado

,UF,Populacao_2010_Compatibilizada,Populacao_2022,Crescimento_Absoluto,Crescimento_Percentual
0,SP,41262199.0,44411238.0,3149039.0,7.63
1,SC,6248436.0,7610361.0,1361925.0,21.80
2,GO,6001789.0,7056495.0,1054706.0,17.57
3,PR,10444526.0,11444380.0,999854.0,9.57
4,MG,19597330.0,20539989.0,942659.0,4.81
5,MT,3035122.0,3658649.0,623527.0,20.54
6,PA,7581051.0,8120131.0,539080.0,7.11
7,AM,3483985.0,3941613.0,457628.0,13.14
8,CE,8451644.0,8794957.0,343313.0,4.06
9,ES,3514952.0,3833712.0,318760.0,9.07


In [5]:
pop_estado.to_csv("populacao_crescimento_por_estado.csv", sep=";", index=False, encoding="utf-8-sig")
print("Arquivo salvo: populacao_crescimento_por_estado.csv")

Arquivo salvo: populacao_crescimento_por_estado.csv


In [6]:
pop_municipio = (
    df.groupby(["UF", "Municipio"])[["Populacao_2010_Compatibilizada", "Populacao_2022"]]
    .sum()
    .reset_index()
)

pop_municipio.head()

,UF,Municipio,Populacao_2010_Compatibilizada,Populacao_2022
0,AC,Acrelândia,12538.0,14021.0
1,AC,Assis Brasil,6072.0,8100.0
2,AC,Brasiléia,21398.0,26000.0
3,AC,Bujari,8471.0,12917.0
4,AC,Capixaba,8798.0,10392.0


In [7]:
pop_municipio["Crescimento_Absoluto"] = pop_municipio["Populacao_2022"] - pop_municipio["Populacao_2010_Compatibilizada"]
pop_municipio["Crescimento_Percentual"] = (
    pop_municipio["Crescimento_Absoluto"] / pop_municipio["Populacao_2010_Compatibilizada"] * 100
).round(2)

pop_municipio = pop_municipio.sort_values("Crescimento_Absoluto", ascending=False).reset_index(drop=True)

pop_municipio.head(20)

,UF,Municipio,Populacao_2010_Compatibilizada,Populacao_2022,Crescimento_Absoluto,Crescimento_Percentual
0,AM,Manaus,1802014.0,2063689.0,261675.0,14.52
1,DF,Brasília,2572159.0,2817381.0,245222.0,9.53
2,SP,São Paulo,11253503.0,11451999.0,198496.0,1.76
3,SP,Sorocaba,586816.0,723682.0,136866.0,23.32
4,GO,Goiânia,1301912.0,1437366.0,135454.0,10.40
5,RR,Boa Vista,284313.0,413486.0,129173.0,45.43
6,SC,Florianópolis,421240.0,537211.0,115971.0,27.53
7,PA,Parauapebas,153908.0,267836.0,113928.0,74.02
8,MS,Campo Grande,786774.0,898100.0,111326.0,14.15
9,PB,João Pessoa,723515.0,833932.0,110417.0,15.26


In [8]:
pop_municipio.to_csv("populacao_crescimento_por_municipio.csv", sep=";", index=False, encoding="utf-8-sig")
print("Arquivo salvo: populacao_crescimento_por_municipio.csv")

Arquivo salvo: populacao_crescimento_por_municipio.csv
